In [1]:
# ==============================================================================
# STEP 0: DATASET SOURCE CONFIGURATION & PATH VERIFICATION
# ==============================================================================

DATASET_META = {
    "dataset_name": "Anemia using Fingernails Image Datasets from Ghana",
    "dataset_url": "https://www.kaggle.com/datasets/kritagyadev/anemia-using-fingernails-image-datasets-from-ghana",
    "kaggle_slug": "kritagyadev/anemia-using-fingernails-image-datasets-from-ghana",
    "kaggle_input_path": "/kaggle/input/datasets/kritagyadev/anemia-using-fingernails-image-datasets-from-ghana",
    "surface": "Nail (Bed/Plate Pallor)",
    "target_classes": ["Anemic", "Non-Anemic"]
}

from pathlib import Path

DATASET_ROOT = Path(DATASET_META["kaggle_input_path"])

# Secondary checks in case Kaggle mounted directly or under alternate slug
if not DATASET_ROOT.exists():
    short_path = Path(f"/kaggle/input/{DATASET_META['kaggle_slug'].split('/')[-1]}")
    if short_path.exists():
        DATASET_ROOT = short_path
    else:
        # Dynamic fallback scan for Ghana / Anemia in /kaggle/input
        matches = [p for p in Path('/kaggle/input').rglob('*') if 'fingernail' in p.name.lower() or 'ghana' in p.name.lower()]
        if matches:
            DATASET_ROOT = matches[0] if matches[0].is_dir() else matches[0].parent

assert DATASET_ROOT.exists(), (
    f" Dataset not found at: {DATASET_META['kaggle_input_path']}\n"
    f"Please attach the dataset from Kaggle: {DATASET_META['dataset_url']}"
)

print(f" Verified: {DATASET_META['dataset_name']}")
print(f" Source URL: {DATASET_META['dataset_url']}")
print(f" Mounted Path: {DATASET_ROOT}")

 Verified: Anemia using Fingernails Image Datasets from Ghana
 Source URL: https://www.kaggle.com/datasets/kritagyadev/anemia-using-fingernails-image-datasets-from-ghana
 Mounted Path: /kaggle/input/datasets/shahrozkhalid11/ghana-fingernail-anemia


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path

for item in Path('/kaggle/input/datasets/shahrozkhalid11/ghana-fingernail-anemia').rglob('*'):
    print(item)

In [ ]:
#Step 1 — Helpers: label + base-name extraction
import re
from pathlib import Path
from PIL import Image
from collections import Counter

FN_ROOT = Path('/kaggle/input/datasets/shahrozkhalid11/ghana-fingernail-anemia/Fingernails')

def extract_label(filename: str):
    """Anything starting with 'Non-' (any casing/typo) is non-anemic; everything else is anemic."""
    if re.match(r'^Non-', filename, re.IGNORECASE):
        return 'Non-Anemic'
    return 'Anemic'

def base_name(filename: str) -> str:
    """Strip a trailing ' (N)' copy-number suffix to recover the original source filename."""
    stem = Path(filename).stem
    return re.sub(r'\s*\(\d+\)\s*$', '', stem).strip()

In [ ]:
#Step 2 — Walk the folder and build the image-level table
import pandas as pd

records = []
for f in FN_ROOT.iterdir():
    if not f.is_file():
        continue
    label = extract_label(f.name)
    src_base = base_name(f.name)
    source_group_id = f"{label}__{src_base}"

    try:
        with Image.open(f) as im:
            width, height = im.size
    except Exception as e:
        width, height = None, None
        print(f"⚠️ Could not open {f}: {e}")

    records.append({
        'source_dataset': 'ghana-fingernail-anemia',
        'image_id': f.stem,
        'file_name': f.name,
        'file_path': str(f),
        'width': width,
        'height': height,
        'Category': label,
        'source_group_id': source_group_id,
    })

df = pd.DataFrame(records)
print(f"Total files: {len(df)}")
print(df['Category'].value_counts())

In [ ]:
#Step 3 — Deduplicate by source group (real unique-image count)
df_dedup = df.drop_duplicates(subset='source_group_id', keep='first').reset_index(drop=True)
print(f"Before dedup: {len(df)} | After dedup: {len(df_dedup)}")
print(df_dedup['Category'].value_counts())
print("Unreadable images:", df_dedup['width'].isna().sum())


In [ ]:
#Step 4 — Check Cross-Class / Subject-Level Collisions
from collections import defaultdict

base_to_classes = defaultdict(set)
for _, row in df_dedup.iterrows():
    base = row['source_group_id'].split('__', 1)[1]
    base_to_classes[base].add(row['Category'])

cross_class = {k: v for k, v in base_to_classes.items() if len(v) > 1}
print(f"Source images mapped to multiple classes: {len(cross_class)}")
if cross_class:
    print("Collisions found:", cross_class)

In [ ]:
#Step 5 — Schema Mapping & Export
anemia_standardized_df = df_dedup.assign(
    **{'Clinical Diagnosis': df_dedup['Category']}
)[['source_dataset', 'image_id', 'file_name', 'width', 'height', 'Category', 'Clinical Diagnosis']]

# Save standardized table
output_path = '/kaggle/working/anemia_standardized_df.csv'
anemia_standardized_df.to_csv(output_path, index=False)

print(f"Saved {len(anemia_standardized_df)} rows to {output_path}.")
anemia_standardized_df.head()

In [ ]:
#Step 6 — Extract patient_id and Verify Patient Count
import re
import pandas as pd

df_anemia = pd.read_csv('/kaggle/working/anemia_standardized_df.csv')

def extract_patient_id(filename: str) -> str:
    """
    Extracts underlying patient ID from patterns like:
    'Anemic-FN-027 (10).png', 'Non-Anrmic-FN-138 (5).png', 'Anmeic-fn-0011 (2).png'
    """
    # Normalize typos and case
    clean_name = filename.replace('Anmeic', 'Anemic').replace('Non-Anrmic', 'Non-Anemic')
    match = re.search(r'(?:Anemic|Non-Anemic)[-_](?:fn|fin|FN|Fin)[-_]?(\d+)', clean_name, re.IGNORECASE)
    if match:
        return f"patient_{match.group(1).zfill(3)}"
    return "unknown"

df_anemia['patient_id'] = df_anemia['file_name'].apply(extract_patient_id)

print(f"Total patches/images: {len(df_anemia)}")
print(f"Total unique patients: {df_anemia['patient_id'].nunique()}")
print(f"Unknown patient IDs: {(df_anemia['patient_id'] == 'unknown').sum()}")
print("\nTop patients by patch count:")
print(df_anemia['patient_id'].value_counts().head(10))

# Verify if any patient is mapped to both Anemic and Non-Anemic
patient_label_clash = df_anemia.groupby('patient_id')['Category'].nunique()
clashes = patient_label_clash[patient_label_clash > 1]
print(f"\nPatients with conflicting diagnostic labels: {len(clashes)}")

In [ ]:
#Step 7 — Update Schema with patient_id and Resolution Stats
# Save updated schema with patient_id included
df_anemia.to_csv('/kaggle/working/anemia_standardized_df.csv', index=False)

print("\nResolution Summary:")
print(df_anemia[['width', 'height']].describe())

In [ ]:
#Step 8 — Fix Patient ID Scoping & Re-verify
import re
import pandas as pd

df_anemia = pd.read_csv('/kaggle/working/anemia_standardized_df.csv')

def extract_scoped_patient_id(row: pd.Series) -> str:
    filename = row['file_name']
    category = row['Category']
    
    clean_name = filename.replace('Anmeic', 'Anemic').replace('Non-Anrmic', 'Non-Anemic')
    match = re.search(r'(?:Anemic|Non-Anemic)[-_](?:fn|fin|FN|Fin)[-_]?(\d+)', clean_name, re.IGNORECASE)
    
    prefix = "anemic" if "non" not in category.lower() else "non_anemic"
    if match:
        return f"{prefix}_pat_{match.group(1).zfill(3)}"
    return f"{prefix}_unknown"

df_anemia['patient_id'] = df_anemia.apply(extract_scoped_patient_id, axis=1)

print(f"Total unique patient IDs: {df_anemia['patient_id'].nunique()}")
patient_label_clash = df_anemia.groupby('patient_id')['Category'].nunique()
clashes = patient_label_clash[patient_label_clash > 1]
print(f"Patients with conflicting diagnostic labels: {len(clashes)}")
print("\nPer-class patient breakdown:")
print(df_anemia.groupby('Category')['patient_id'].nunique())

In [ ]:
#Step 9 — Low-Resolution Filtering & Save
# Flag very small patches (< 32px on either side)
small_patches = df_anemia[(df_anemia['width'] < 32) | (df_anemia['height'] < 32)]
print(f"Patches smaller than 32x32: {len(small_patches)}")

# Save final standardized anemia table
df_anemia.to_csv('/kaggle/working/anemia_standardized_df.csv', index=False)
print("Saved clean anemia standardized dataframe.")